In [ ]:
import pandas as pd
import numpy as np
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, roc_auc_score, accuracy_score, precision_score, recall_score
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from transformers import EvalPrediction
from datasets import Dataset, DatasetDict

In [ ]:
# ---- 1. Load and Prepare the Data

df = pd.read_excel('ground_truth_LLM804.xlsx')

# Define the text column and extract the label columns
text_col = 'Headline_GER'
label_cols = [col for col in df.columns if col != text_col]

# Convert the 14 label columns into a single list of floats for each row
# (Float is required for BCEWithLogitsLoss in multi-label classification)
df['labels'] = df[label_cols].values.astype(np.float32).tolist()

# Keep only the necessary columns for training
df_clean = df[[text_col, 'labels']].copy()

In [ ]:
# ---- 2. Apply the Train / Dev / Test Splits

# Initial 80/20 Train-Rest split
train_df, rest_df = train_test_split(df_clean, test_size=0.20, random_state=42)

# Splitting the Rest (20%) into 60% Dev and 40% Test
dev_df, test_df = train_test_split(rest_df, test_size=0.40, random_state=42)

print(f"Train size: {len(train_df)}")
print(f"Dev size:   {len(dev_df)}")
print(f"Test size:  {len(test_df)}")

# Convert Pandas DataFrames into Hugging Face Datasets
dataset = DatasetDict({
    'train': Dataset.from_pandas(train_df, preserve_index=False),
    'dev': Dataset.from_pandas(dev_df, preserve_index=False),
    'test': Dataset.from_pandas(test_df, preserve_index=False)
})

In [ ]:
tokenized_datasets = dataset.map(tokenize_function, batched=True)

In [ ]:
# Load the model with the modified configuration
model_LLM = AutoModelForSequenceClassification.from_pretrained(
    model_id,
    config=config
)

In [ ]:
trainer = Trainer(
    model=model_LLM,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["dev"],
    compute_metrics=compute_metrics
)

In [ ]:
# ---- 7. Train and Evaluate

# Execute the training loop
trainer.train()

# Final evaluation against the unseen test set
print("\n--- Final Test Set Evaluation ---")
test_results = trainer.evaluate(tokenized_datasets["test"])
print(test_results)